### Import Library

In [64]:
pip install pandas numpy matplotlib seaborn nltk gensim scikit-learn jupyter ipykernel streamlit

Note: you may need to restart the kernel to use updated packages.


In [65]:
# Data Manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Text Processing
import re
import string

# Utility
import warnings
warnings.filterwarnings("ignore")

# Display Settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

# Random Seed
RANDOM_STATE = 42

### Load Dataset

In [66]:
df = pd.read_csv("../dataset/job_description.csv")

print(f"Dataset Shape : {df.shape}")
df.head()

Dataset Shape : (607, 5)


,Job Field,Job Title,Job Description,Key Responsibilities,Required Skills & Qualifications
0,Retail Banking,Bank Teller,Front-line customer service professional responsible for processing daily financial transactions and addressing basic customer inquiries.,"• Process deposits, withdrawals, and loan payments.<br>• Balance a cash drawer daily.<br>• Answer account inquiries and identify referral opportunities.","• High school diploma.<br>• Strong cash handling, accuracy, and customer service skills."
1,Retail Banking,Lead Teller,"A senior teller who processes transactions while also providing oversight, coaching, and vault management support to the teller line.",• Perform all teller duties.<br>• Manage and balance the bank vault.<br>• Coach junior tellers and handle escalated customer issues.,"• 2+ years of teller experience.<br>• Leadership potential, advanced problem-solving, and cash vault management skills."
2,Retail Banking,Personal Banker,A sales and service professional in a branch responsible for opening new accounts and selling bank products like loans and credit cards.,• Open new checking/savings accounts and CDs.<br>• Consult with customers to identify needs.<br>• Process consumer loan and credit card applications.<br>• Meet established sales and referral goals.,"• Bachelor's degree preferred.<br>• Strong sales, communication, and relationship-building skills."
3,Retail Banking,Relationship Banker,"A more advanced Personal Banker role focused on developing deeper, long-term relationships with a portfolio of branch clients.","• Proactively manage and grow a portfolio of consumer clients.<br>• Provide holistic financial advice.<br>• Cross-sell a full range of bank products (loans, investments, etc.).","• Bachelor's in Finance/Business.<br>• Proven sales record, financial acumen, and CRM skills."
4,Retail Banking,Small Business Banker,"A specialized banker, often located within a branch, who focuses exclusively on the needs of small business clients.","• Open new business checking accounts.<br>• Originate small business loans (e.g., SBA loans, lines of credit).<br>• Sell treasury/cash management services to small businesses.",• Bachelor's in Business/Finance.<br>• B2B sales skills and a strong understanding of business financial statements.


### Dataset Overview

In [67]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 607 entries, 0 to 606
Data columns (total 5 columns):
 #   Column                            Non-Null Count  Dtype
---  ------                            --------------  -----
 0   Job Field                         607 non-null    str  
 1   Job Title                         607 non-null    str  
 2   Job Description                   607 non-null    str  
 3   Key Responsibilities              607 non-null    str  
 4   Required Skills & Qualifications  607 non-null    str  
dtypes: str(5)
memory usage: 393.5 KB


In [68]:
# Missing Values

missing = (
    df.isnull()
      .sum()
      .sort_values(ascending=False)
      .rename("Missing Values")
)

missing

Job Field                           0
Job Title                           0
Job Description                     0
Key Responsibilities                0
Required Skills & Qualifications    0
Name: Missing Values, dtype: int64

In [69]:
# Duplicate Rows

duplicates = df.duplicated().sum()

print(f"Duplicate Rows : {duplicates}")

Duplicate Rows : 26


In [70]:
# Unique Values

for col in df.columns:
    print(f"{col:<35} : {df[col].nunique()}")

Job Field                           : 91
Job Title                           : 579
Job Description                     : 578
Key Responsibilities                : 578
Required Skills & Qualifications    : 578


In [71]:
df.describe(include="object")

,Job Field,Job Title,Job Description,Key Responsibilities,Required Skills & Qualifications
count,607,607,607,607,607
unique,91,579,578,578,578
top,Technology Consulting,Grant Writer (Arts),"A specialized writer who researches and writes grant proposals to foundations, corporations, and government agencies (like the NEA) to secure funding for an arts organization.","• Research and identify potential grant funding opportunities.<br>• Write clear, persuasive grant applications and proposals that match the funder's goals.<br>• Gather all supporting documentation (budgets, project plans).<br>• Manage grant deadlines and write final reports on how funds were used.",• Exceptional persuasive writing skills.<br>• Strong research and project management skills.<br>• Ability to write detailed budgets.<br>• Experience in non-profit fundraising.
freq,22,3,3,3,3


In [72]:
df.sample(5, random_state=RANDOM_STATE)

,Job Field,Job Title,Job Description,Key Responsibilities,Required Skills & Qualifications
563,Niche: Public Sector,Public Sector Financials Consultant,"Advises government agencies on budgeting, fund accounting, and compliance with governmental accounting standards (GASB).",• Help government clients manage their budgets and funds.<br>• Ensure compliance with GASB standards and federal grant rules.<br>• Assist with the preparation of the Comprehensive Annual Financial Report (CAFR).<br>• Implement public sector ERP systems (like Oracle PS).,"• Skills: Fund accounting, GASB, government budgeting, federal regulations.<br>• Qualifications: CPA with government experience; CGFM (Certified Government Financial Manager)."
289,Healthcare Admin/Mgmt,Risk Manager (Healthcare),"A specialist who identifies and mitigates any risk that could lead to patient harm or financial loss (e.g., lawsuits, injuries) for the hospital.","• Investigate patient safety incidents, medication errors, and patient falls.<br>• Analyze incident data to identify trends and implement preventative strategies.<br>• Manage malpractice claims and liaise with legal counsel.<br>• Oversee patient safety programs.","• Skills: Risk assessment, investigative skills, knowledge of healthcare law, data analysis.<br>• Qualifications: Clinical background (RN or JD) preferred. Certification in healthcare risk management (CPHRM)."
76,Investment Banking,Prime Brokerage Associate,"Works in a specialized group that provides a bundle of services to hedge fund clients (e.g., lending them cash/securities, trade execution, and custody).",• Act as the relationship manager for hedge fund clients.<br>• Facilitate leverage (margin lending) for the fund.<br>• Coordinate trade settlement and cash management for the fund.,"• Bachelor's/MBA.<br>• Deep knowledge of hedge fund operations, risk, and derivatives."
78,Wealth Management,Family Office Relationship Manager,"Services the most elite clients (Single Family Offices), coordinating all bank services for families typically with $100M+ in assets.","• Act as the ""quarterback"" for all bank services for a single family.<br>• Coordinate private banking, custom credit, and institutional-level investing.<br>• Manage generational wealth transfer and philanthropic planning.",• MBA/CFA/JD.<br>• Decades of experience serving UHNW individuals.
182,Industrial,Industrial Engineer (IE),"An engineer focused on optimizing complex systems and processes to eliminate waste. They make processes faster, safer, more efficient, and more cost-effective.",• Analyze manufacturing or service processes to find bottlenecks.<br>• Use statistical analysis and modeling to improve efficiency.<br>• Design factory floor layouts for optimal workflow.<br>• Implement Lean Manufacturing and Six Sigma principles.,• BS in Industrial & Systems Engineering (ISE).


In [73]:
df.columns

Index(['Job Field', 'Job Title', 'Job Description', 'Key Responsibilities',
       'Required Skills & Qualifications'],
      dtype='str')

In [74]:
sample = df.iloc[0]

for col in df.columns:
    print("=" * 80)
    print(col)
    print(sample[col])

Job Field
Retail Banking
Job Title
Bank Teller
Job Description
Front-line customer service professional responsible for processing daily financial transactions and addressing basic customer inquiries.
Key Responsibilities
• Process deposits, withdrawals, and loan payments.<br>• Balance a cash drawer daily.<br>• Answer account inquiries and identify referral opportunities.
Required Skills & Qualifications
• High school diploma.<br>• Strong cash handling, accuracy, and customer service skills.


### Data Cleaning

In [75]:
# HTML and Text Cleaning

import re

def clean_raw_text(text):
    if pd.isna(text):
        return ""

    # Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)

    # Replace bullet symbols
    text = text.replace("•", " ")

    # Replace newlines
    text = text.replace("\n", " ")

    # Remove multiple spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [76]:
columns = [
    "Job Description",
    "Key Responsibilities",
    "Required Skills & Qualifications"
]

for col in columns:
    df[col] = df[col].apply(clean_raw_text)

In [77]:
sample = df.iloc[0]

for col in columns:
    print("=" * 80)
    print(col)
    print(sample[col])

Job Description
Front-line customer service professional responsible for processing daily financial transactions and addressing basic customer inquiries.
Key Responsibilities
Process deposits, withdrawals, and loan payments. Balance a cash drawer daily. Answer account inquiries and identify referral opportunities.
Required Skills & Qualifications
High school diploma. Strong cash handling, accuracy, and customer service skills.


In [78]:
# Remove Duplicate Rows

print(f"Shape before : {df.shape}")

df = df.drop_duplicates().reset_index(drop=True)

print(f"Shape after  : {df.shape}")

Shape before : (607, 5)
Shape after  : (581, 5)


In [79]:
print(df.duplicated().sum())

0


In [80]:
# Create Career Text

def combine_text(row):
    parts = [
        row["Job Title"],
        row["Job Description"],
        row["Key Responsibilities"],
        row["Required Skills & Qualifications"]
    ]

    # Hapus bagian kosong
    parts = [str(part).strip() for part in parts if str(part).strip()]

    return " ".join(parts)

df["career_text"] = df.apply(combine_text, axis=1)

In [81]:
print(df["career_text"].iloc[0])

Bank Teller Front-line customer service professional responsible for processing daily financial transactions and addressing basic customer inquiries. Process deposits, withdrawals, and loan payments. Balance a cash drawer daily. Answer account inquiries and identify referral opportunities. High school diploma. Strong cash handling, accuracy, and customer service skills.


### Text Preprocessing

In [82]:
import nltk
import re

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("punkt")
nltk.download('punkt_tab')
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package punkt to /home/muriz/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/muriz/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/muriz/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/muriz/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/muriz/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [83]:
# Initialize NLP Components

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

In [84]:
# Text Preprocessing Function

def preprocess_text(text):

    # Lowercase
    text = text.lower()

    # Ubah tanda hubung menjadi spasi
    text = text.replace("-", " ")

    # Hapus karakter selain huruf, angka, dan spasi
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Hapus spasi berlebih
    text = re.sub(r"\s+", " ", text).strip()

    # Tokenization
    tokens = word_tokenize(text)

    # Stopword Removal
    tokens = [
        word for word in tokens
        if word not in stop_words
    ]

    # Lemmatization
    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
    ]

    return " ".join(tokens)

In [85]:
sample = df["career_text"].iloc[0]

print("BEFORE\n")
print(sample)

print("\n" + "="*100 + "\n")

print("AFTER\n")
print(preprocess_text(sample))

BEFORE

Bank Teller Front-line customer service professional responsible for processing daily financial transactions and addressing basic customer inquiries. Process deposits, withdrawals, and loan payments. Balance a cash drawer daily. Answer account inquiries and identify referral opportunities. High school diploma. Strong cash handling, accuracy, and customer service skills.


AFTER

bank teller front line customer service professional responsible processing daily financial transaction addressing basic customer inquiry process deposit withdrawal loan payment balance cash drawer daily answer account inquiry identify referral opportunity high school diploma strong cash handling accuracy customer service skill


In [86]:
# Apply Preprocessing

df["processed_text"] = df["career_text"].apply(preprocess_text)

In [87]:
df[["career_text", "processed_text"]].head()

,career_text,processed_text
0,"Bank Teller Front-line customer service professional responsible for processing daily financial transactions and addressing basic customer inquiries. Process deposits, withdrawals, and loan payments. Balance a cash drawer daily. Answer account inquiries and identify referral opportunities. High school diploma. Strong cash handling, accuracy, and customer service skills.",bank teller front line customer service professional responsible processing daily financial transaction addressing basic customer inquiry process deposit withdrawal loan payment balance cash drawer daily answer account inquiry identify referral opportunity high school diploma strong cash handling accuracy customer service skill
1,"Lead Teller A senior teller who processes transactions while also providing oversight, coaching, and vault management support to the teller line. Perform all teller duties. Manage and balance the bank vault. Coach junior tellers and handle escalated customer issues. 2+ years of teller experience. Leadership potential, advanced problem-solving, and cash vault management skills.",lead teller senior teller process transaction also providing oversight coaching vault management support teller line perform teller duty manage balance bank vault coach junior teller handle escalated customer issue 2 year teller experience leadership potential advanced problem solving cash vault management skill
2,"Personal Banker A sales and service professional in a branch responsible for opening new accounts and selling bank products like loans and credit cards. Open new checking/savings accounts and CDs. Consult with customers to identify needs. Process consumer loan and credit card applications. Meet established sales and referral goals. Bachelor's degree preferred. Strong sales, communication, and relationship-building skills.",personal banker sale service professional branch responsible opening new account selling bank product like loan credit card open new checking saving account cd consult customer identify need process consumer loan credit card application meet established sale referral goal bachelor degree preferred strong sale communication relationship building skill
3,"Relationship Banker A more advanced Personal Banker role focused on developing deeper, long-term relationships with a portfolio of branch clients. Proactively manage and grow a portfolio of consumer clients. Provide holistic financial advice. Cross-sell a full range of bank products (loans, investments, etc.). Bachelor's in Finance/Business. Proven sales record, financial acumen, and CRM skills.",relationship banker advanced personal banker role focused developing deeper long term relationship portfolio branch client proactively manage grow portfolio consumer client provide holistic financial advice cross sell full range bank product loan investment etc bachelor finance business proven sale record financial acumen crm skill
4,"Small Business Banker A specialized banker, often located within a branch, who focuses exclusively on the needs of small business clients. Open new business checking accounts. Originate small business loans (e.g., SBA loans, lines of credit). Sell treasury/cash management services to small businesses. Bachelor's in Business/Finance. B2B sales skills and a strong understanding of business financial statements.",small business banker specialized banker often located within branch focus exclusively need small business client open new business checking account originate small business loan e g sba loan line credit sell treasury cash management service small business bachelor business finance b2b sale skill strong understanding business financial statement


In [88]:
df["processed_text"].str.split().str.len().describe()

count    581.000000
mean      59.208262
std        9.696053
min       36.000000
25%       52.000000
50%       60.000000
75%       66.000000
max       88.000000
Name: processed_text, dtype: float64

### Feature Engineering

In [89]:
corpus = df["processed_text"].tolist()

len(corpus)

581

In [90]:
corpus[:3]

['bank teller front line customer service professional responsible processing daily financial transaction addressing basic customer inquiry process deposit withdrawal loan payment balance cash drawer daily answer account inquiry identify referral opportunity high school diploma strong cash handling accuracy customer service skill',
 'lead teller senior teller process transaction also providing oversight coaching vault management support teller line perform teller duty manage balance bank vault coach junior teller handle escalated customer issue 2 year teller experience leadership potential advanced problem solving cash vault management skill',
 'personal banker sale service professional branch responsible opening new account selling bank product like loan credit card open new checking saving account cd consult customer identify need process consumer loan credit card application meet established sale referral goal bachelor degree preferred strong sale communication relationship building

In [91]:
tokenized_corpus = [
    text.split()
    for text in corpus
]

tokenized_corpus[:2]

[['bank',
  'teller',
  'front',
  'line',
  'customer',
  'service',
  'professional',
  'responsible',
  'processing',
  'daily',
  'financial',
  'transaction',
  'addressing',
  'basic',
  'customer',
  'inquiry',
  'process',
  'deposit',
  'withdrawal',
  'loan',
  'payment',
  'balance',
  'cash',
  'drawer',
  'daily',
  'answer',
  'account',
  'inquiry',
  'identify',
  'referral',
  'opportunity',
  'high',
  'school',
  'diploma',
  'strong',
  'cash',
  'handling',
  'accuracy',
  'customer',
  'service',
  'skill'],
 ['lead',
  'teller',
  'senior',
  'teller',
  'process',
  'transaction',
  'also',
  'providing',
  'oversight',
  'coaching',
  'vault',
  'management',
  'support',
  'teller',
  'line',
  'perform',
  'teller',
  'duty',
  'manage',
  'balance',
  'bank',
  'vault',
  'coach',
  'junior',
  'teller',
  'handle',
  'escalated',
  'customer',
  'issue',
  '2',
  'year',
  'teller',
  'experience',
  'leadership',
  'potential',
  'advanced',
  'problem',
 

### Word Embedding

In [92]:
from gensim.models import Word2Vec

In [93]:
# Train Word2Vec

word2vec_model = Word2Vec(
    sentences=tokenized_corpus,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    epochs=30,
    seed=RANDOM_STATE
)

In [94]:
len(word2vec_model.wv)

4868

In [95]:
word2vec_model.wv["python"]

array([ 0.4141683 , -0.03405257, -0.77765816, -0.19623582, -0.94532245,
        0.24953862,  0.18659246,  0.14403328,  0.47313517,  0.70128703,
       -0.89519066,  0.46872967, -0.3763819 ,  0.07647805,  0.75260305,
       -0.02800759, -0.5417081 , -0.2800053 , -0.51008093, -0.563932  ,
        0.15113391,  0.44253808, -0.27047262,  0.17544161,  0.1801416 ,
       -0.3626976 , -0.9898723 ,  0.48087746, -0.26259336,  0.24058414,
        0.5627914 ,  0.02273735,  0.01772627, -0.36526147,  0.57974905,
       -0.58449847,  0.20234053, -0.9025531 ,  0.23794624,  0.73000246,
        0.23921412,  0.4349439 ,  0.94836044, -0.50626487,  0.14725752,
       -0.3832512 , -0.05942588,  0.38302085,  0.20946635,  0.03334994,
       -0.06421155, -0.48223248, -0.24572603,  0.7893973 ,  0.33225045,
       -0.11499304,  0.1745929 , -0.6003123 ,  0.11885961,  0.03895406,
       -0.21258084,  0.46952763, -0.20241222,  0.7547824 , -0.45229635,
        0.1310797 , -0.06058297, -0.6816619 , -0.17549011,  0.22

In [96]:
word2vec_model.wv["bank"]

array([ 0.5504241 , -0.8167192 , -0.4469449 , -0.2304303 , -0.29017368,
       -0.90828085,  0.63073134, -0.00170886, -0.16313508, -0.55770034,
       -0.15886967, -0.33343905, -0.43670383,  0.49830085, -0.3744234 ,
       -0.10576987,  0.03032809, -0.0121037 , -1.0564172 , -0.00528269,
        0.05965509,  0.97383803,  0.25640538,  0.15609145, -0.3539966 ,
        0.7053524 , -0.22414047,  0.15463886, -0.12909222,  0.0876745 ,
       -0.04197485,  0.04794237,  0.17696339, -0.05272932,  0.05491579,
        0.05791815,  0.06098772, -0.1344454 ,  0.31969556, -0.32520074,
        0.01963723,  0.4285657 ,  0.0595798 ,  0.2057578 ,  0.538613  ,
       -0.54771197,  0.38677934,  0.83582604,  0.20993176, -0.03769967,
        0.27573955, -0.15817668, -0.8378052 ,  0.5259044 ,  0.27490467,
        0.13081107,  0.47530815,  1.1635554 ,  0.12645881,  0.3009069 ,
       -0.09246639,  0.25051504, -0.1220989 , -0.23614523, -0.9314758 ,
       -0.17868641,  0.04625576, -0.37583417,  0.37506622,  0.14

In [97]:
word2vec_model.wv.most_similar("bank")

[('lending', 0.7031295895576477),
 ('send', 0.6713405847549438),
 ('anti', 0.6702603697776794),
 ('discriminating', 0.6700484752655029),
 ('designated', 0.6698431968688965),
 ('cra', 0.6679407954216003),
 ('appetite', 0.6574029326438904),
 ('defaulting', 0.653445839881897),
 ('secrecy', 0.6494787335395813),
 ('mrm', 0.6444019675254822)]

### Document Embedding

In [98]:
def document_vector(text):

    words = text.split()

    vectors = [
        word2vec_model.wv[word]
        for word in words
        if word in word2vec_model.wv
    ]

    if len(vectors) == 0:
        return np.zeros(word2vec_model.vector_size)

    return np.mean(vectors, axis=0)

In [99]:
df["career_vector"] = df["processed_text"].apply(document_vector)

In [100]:
print(df["career_vector"].iloc[0].shape)

(100,)


In [101]:
df["career_vector"].head()

0                    [0.3488467, -0.13806456, 0.052217524, 0.3617026, 0.03893726, -0.16595839, 0.19192317, 0.117937356, 0.009519061, -0.020162102, -0.25876272, 0.0704815, -0.06303635, 0.45320633, -0.10819854, -0.37492868, 0.38631663, -0.12529571, -0.47469833, -0.24959178, 0.29992068, 0.5312609, 0.5377399, 0.19133979, -0.1316591, 0.1518334, -0.20921259, 0.1936739, -0.09125817, 0.12076494, -0.13142487, -0.009150116, 0.10906473, -0.27784607, 0.02113083, 0.05528506, -0.04328522, -0.45022923, -0.056108747, 0.17561217, 0.11523203, 0.056968536, 0.3681905, 0.22709589, 0.28265497, -0.23950565, 0.5037576, 0.20875119, 0.3114803, 0.08430324, -0.18199922, -0.06474438, -0.2819998, -0.03203805, 0.246503, 0.022625305, -0.113997795, 0.34081912, 0.21948652, 0.05453688, -0.047131393, -0.1399141, -0.012250097, 0.028718337, -0.19927223, -0.034442935, 0.08165156, -0.047382638, 0.22241907, 0.20758957, -0.20317031, 0.01343045, 0.03198423, 0.3099925, 0.004504816, -0.15172969, 0.031603258, -0.042905506, -0.4859

### Career Recommendation

In [102]:
from sklearn.metrics.pairwise import cosine_similarity

In [103]:
def get_user_vector(user_input):

    processed = preprocess_text(user_input)

    vector = document_vector(processed)

    return processed, vector

In [104]:
def recommend_career(user_input, top_k=3):
    """
    Recommend careers based on user self-description.

    Parameters
    ----------
    user_input : str
        User self-description.
    top_k : int
        Number of recommendations to return.

    Returns
    -------
    pandas.DataFrame
        Top career recommendations sorted by similarity.
    """

    # Preprocess user input
    processed_text, user_vector = get_user_vector(user_input)

    # Calculate cosine similarity
    similarities = cosine_similarity(
        [user_vector],
        list(df["career_vector"])
    )[0]

    # Copy dataframe
    result = df.copy()

    # Add similarity score (percentage)
    result["Similarity (%)"] = (similarities * 100).round(2)

    # Sort by similarity
    result = result.sort_values(
        by="Similarity (%)",
        ascending=False
    )

    # Keep only top-k recommendations
    result = result.head(top_k).reset_index(drop=True)

    # Add ranking
    result.index = result.index + 1
    result.index.name = "Rank"

    return result[
        [
            "Job Title",
            "Job Field",
            "Job Description",
            "Required Skills & Qualifications",
            "Similarity (%)"
        ]
    ]

In [105]:
user_profile = """
I enjoy analyzing data, creating dashboards,
working with SQL, Python,
and machine learning.
"""

recommend_career(user_profile)

,Job Title,Job Field,Job Description,Required Skills & Qualifications,Similarity (%)
Rank,,,,,
1,Data Science Consultant,Technology Consulting,"A hands-on builder (Data Scientist) who works on client projects to build, test, and deploy machine learning models to solve specific business problems.","Skills: Python/R, SQL, machine learning, statistics, data visualization, communication. Qualifications: Master's or PhD in a quantitative field (CompSci, Statistics, Econ).",93.820000
2,Data & Analytics Consultant,Technology Consulting,"Helps clients leverage their data. Focuses on building business intelligence (BI) dashboards, setting up data visualization tools, and pulling insights from data.","Skills: Tableau/Power BI (expert-level), SQL, data warehousing, communication, data storytelling. Qualifications: Degree in analytics, business, or CS. Strong portfolio of dashboards.",92.730003
3,Data Migration Consultant,Technology Consulting,"A highly technical specialist who plans and executes the complex, high-risk project of moving data from a client's legacy system(s) to a new system (like a new ERP or cloud database).","Skills: ETL processes, SQL, scripting (Python), data modeling, extreme attention to detail. Qualifications: Technical background in data management; experience on large transformation projects.",91.599998


In [106]:
user_profile = """
I enjoy helping customers,
handling financial transactions,
and working in banking.
"""

recommend_career(user_profile)

,Job Title,Job Field,Job Description,Required Skills & Qualifications,Similarity (%)
Rank,,,,,
1,Bank Teller,Retail Banking,Front-line customer service professional responsible for processing daily financial transactions and addressing basic customer inquiries.,"High school diploma. Strong cash handling, accuracy, and customer service skills.",90.760002
2,Loan Servicing Specialist,Bank Operations,"A ""post-closing"" operations role that manages a loan after it has closed, handling payments, taxes, and customer service inquiries.","High school diploma. Strong bookkeeping, data entry, and customer service skills.",88.580002
3,Universal Banker,Retail Banking,A hybrid branch role that combines the duties of both a Teller (transactions) and a Personal Banker (sales and account opening).,"High school diploma/Bachelor's. Strong skills in both customer service, cash handling, and sales.",88.500000


In [107]:
user_profile = """
I enjoy designing user interfaces,
wireframes,
and prototypes.
"""

recommend_career(user_profile)

,Job Title,Job Field,Job Description,Required Skills & Qualifications,Similarity (%)
Rank,,,,,
1,UX Designer (User Experience),Design (Digital),"Focuses on the overall feel of a digital product. They are responsible for making a website or app functional, intuitive, and enjoyable for the user to interact with.","Wireframing (Figma, Sketch), prototyping. User research methods. Empathy and analytical thinking. Portfolio of UX case studies.",88.800003
2,Product Designer (Digital),Design (Digital),"A strategic role, often in tech, that combines UX design, UI design, and user research. They are responsible for the entire experience of a digital product, from initial research to final visual design.",Mastery of the entire design process (UX/UI). Strategic thinking and problem-solving. Proficiency in Figma/Sketch. Portfolio of product case studies.,88.250000
3,Frontend Engineer,Software,"A software engineer specializing in the ""client-side""—the visual elements of a website or application that the user directly interacts with in their browser.","BS in CS. Expertise in JavaScript, HTML5, CSS3, and modern JS frameworks.",84.949997


### Evaluation

In [108]:
# Functional Testing

test_profile = """
I enjoy analyzing data,
building machine learning models,
working with Python and SQL.
"""

recommend_career(test_profile)

,Job Title,Job Field,Job Description,Required Skills & Qualifications,Similarity (%)
Rank,,,,,
1,Data Science Consultant,Technology Consulting,"A hands-on builder (Data Scientist) who works on client projects to build, test, and deploy machine learning models to solve specific business problems.","Skills: Python/R, SQL, machine learning, statistics, data visualization, communication. Qualifications: Master's or PhD in a quantitative field (CompSci, Statistics, Econ).",92.750000
2,"Quantitative Analyst (""Quant"")",Investment Banking,"Designs and implements complex mathematical models that are used to price securities, assess risk, or execute algorithmic trading strategies.","PhD or Master's in a highly quantitative field (Math, Physics, Stats). Elite programming (C++, Python) and mathematical skills.",89.519997
3,Data & Analytics Consultant,Technology Consulting,"Helps clients leverage their data. Focuses on building business intelligence (BI) dashboards, setting up data visualization tools, and pulling insights from data.","Skills: Tableau/Power BI (expert-level), SQL, data warehousing, communication, data storytelling. Qualifications: Degree in analytics, business, or CS. Strong portfolio of dashboards.",89.070000


Sistem berhasil menghasilkan tiga rekomendasi karier teratas berdasarkan deskripsi diri pengguna. Rekomendasi-rekomendasi tersebut diurutkan berdasarkan kemiripan kosinus antara vektor dokumen pengguna dan vektor-vektor karier yang dihasilkan menggunakan embedding Word2Vec.

In [109]:
# Multiple Test Scenarios

test_cases = [
    {
        "Profile": "Data Scientist",
        "Description": """
        I enjoy machine learning,
        Python,
        SQL,
        statistics,
        and data visualization.
        """
    },

    {
        "Profile": "UI Designer",
        "Description": """
        I enjoy designing user interfaces,
        creating wireframes,
        prototypes,
        and improving user experience.
        """
    },

    {
        "Profile": "Backend Developer",
        "Description": """
        I build REST APIs,
        databases,
        Docker,
        Python,
        and cloud services.
        """
    },

    {
        "Profile": "Cybersecurity",
        "Description": """
        I enjoy network security,
        penetration testing,
        risk assessment,
        and vulnerability analysis.
        """
    },

    {
        "Profile": "Accountant",
        "Description": """
        I enjoy financial reporting,
        taxation,
        auditing,
        budgeting,
        and accounting.
        """
    },

    {
        "Profile": "Banking",
        "Description": """
        I enjoy customer service,
        banking,
        cash handling,
        deposits,
        and financial transactions.
        """
    },

    {
        "Profile": "Marketing",
        "Description": """
        I enjoy SEO,
        digital marketing,
        social media,
        branding,
        and advertising.
        """
    },

    {
        "Profile": "Project Manager",
        "Description": """
        I enjoy planning projects,
        managing teams,
        budgeting,
        scheduling,
        and communication.
        """
    },

    {
        "Profile": "HR",
        "Description": """
        I enjoy recruitment,
        interviewing,
        employee engagement,
        onboarding,
        and performance evaluation.
        """
    },

    {
        "Profile": "Business Analyst",
        "Description": """
        I enjoy business analysis,
        SQL,
        dashboards,
        documentation,
        and stakeholder communication.
        """
    }
]

In [110]:
# Multiple Test Scenarios

evaluation_results = []

for case in test_cases:

    recommendation = recommend_career(
        case["Description"],
        top_k=1
    )

    evaluation_results.append({

        "Input Profile": case["Profile"],

        "Recommended Career":
            recommendation.iloc[0]["Job Title"],

        "Similarity (%)":
            recommendation.iloc[0]["Similarity (%)"]

    })

evaluation_df = pd.DataFrame(evaluation_results)

evaluation_df

,Input Profile,Recommended Career,Similarity (%)
0,Data Scientist,Data Science Consultant,91.320000
1,UI Designer,UX Designer (User Experience),91.730003
2,Backend Developer,Cloud Implementation Consultant,92.370003
3,Cybersecurity,Cybersecurity Risk Analyst,90.449997
4,Accountant,Public Sector Financials Consultant,89.099998
5,Banking,Bank Teller,93.500000
6,Marketing,Digital Marketing Consultant,91.160004
7,Project Manager,Engagement Manager / Project Lead,91.019997
8,HR,Branch Manager,88.809998
9,Business Analyst,Data & Analytics Consultant,89.449997


In [111]:
correct = [
    True,
    True,
    True,
    True,
    True,
    True,
    True,
    True,
    False,
    True
]

accuracy = sum(correct) / len(correct)

print(f"Recommendation Accuracy: {accuracy:.2%}")

Recommendation Accuracy: 90.00%


In [112]:
user_profile = """
I enjoy developing web applications,
building REST APIs,
working with Docker,
Python,
and cloud services.
"""

recommend_career(user_profile)

,Job Title,Job Field,Job Description,Required Skills & Qualifications,Similarity (%)
Rank,,,,,
1,Cloud Engineer,Software,"An IT/engineering role focused on designing, deploying, and managing applications and infrastructure on cloud computing platforms (AWS, Azure, GCP).","BS in CS or IT. Cloud certifications (e.g., AWS Certified Solutions Architect) are critical.",92.510002
2,Cloud Implementation Consultant,Technology Consulting,"A hands-on technical expert who actually builds, configures, and deploys solutions on cloud platforms for a client.","Skills: Expert-level AWS/Azure/GCP, Terraform/Ansible, scripting (Python), networking, security. Qualifications: Professional-level cloud certifications (e.g., AWS Solutions Architect Pro).",91.269997
3,Backend Engineer,Software,"A software engineer specializing in the ""server-side""—the database, servers, and APIs that power the application behind the scenes.","BS in CS. Expertise in backend languages (e.g., Java, Python, Node.js, Go) and database design.",90.769997


### Save Model

In [113]:
import pickle

In [114]:
word2vec_model.save("../models/word2vec.model")

In [115]:
with open("../models/career_dataset.pkl", "wb") as f:
    pickle.dump(df, f)

### Discussion

#### Hasil rekomendasi menunjukkan bahwa sistem yang diusulkan mampu mengidentifikasi karier yang sangat sesuai dengan deskripsi diri pengguna. Profil-profil yang berkaitan dengan ilmu data, perbankan, dan desain antarmuka pengguna secara konsisten menghasilkan rekomendasi dengan skor kesamaan kosinus yang tinggi. Hal ini menunjukkan bahwa embedding Word2Vec berhasil menangkap hubungan semantik antara deskripsi pengguna dan deskripsi pekerjaan yang terdapat dalam kumpulan data.

#### Karena embedding tersebut hanya dilatih menggunakan dataset deskripsi pekerjaan yang tersedia, representasi semantiknya terbatas pada kosakata yang terdapat dalam korpus tersebut. Penggunaan model embedding yang telah dilatih sebelumnya dengan skala lebih besar atau embedding kontekstual dapat semakin meningkatkan kualitas rekomendasi.